# Gold — Data Quality Checks

Runs assertions against the Gold layer after every pipeline execution.
Each check raises immediately on failure so the pipeline halts and alerts before bad data reaches the Semantic Model.

**Runs after:** `gold_fact_monthly_economics`, `gold_dim_date`, `gold_dim_indicator`, `gold_dim_source`, `gold_dim_geography`  

| Check | Table | Description |
|---|---|---|
| No null keys or values | `fact_monthly_economics` | Core fact columns must never be null |
| No duplicate grain | `fact_monthly_economics` | `(month_key, indicator_key)` must be unique |
| Referential integrity — date | `fact_monthly_economics` | Every `date_key` must exist in `dim_date` |
| Referential integrity — indicator | `fact_monthly_economics` | Every `indicator_key` must exist in `dim_indicator` |
| Referential integrity — source | `fact_monthly_economics` | Every `source_key` must exist in `dim_source` |
| Referential integrity — geography | `fact_monthly_economics` | Every `geography_key` must exist in `dim_geography` |
| Value range — policy rate | `fact_monthly_economics` | Must be between 0% and 30% |
| Value range — CPI | `fact_monthly_economics` | Must be between -10% and 100% |
| Value range — exchange rates | `fact_monthly_economics` | ISK/USD and ISK/EUR must be positive |
| Minimum row count | `fact_monthly_economics` | Must have at least 7 rows per month covered |
| All 7 indicators present | `fact_monthly_economics` | Every indicator must appear in the latest month |
| dim_date coverage | `dim_date` | Must cover the full fact date range |

In [ ]:
# ── DQ helper ──────────────────────────────────────────────────────────────
# Each check passes a SQL query that returns a count of failing rows.
# Zero failing rows = pass. Any failing rows = immediate halt.

passed = []
failed = []

def assert_check(name, sql):
    failing_rows = spark.sql(sql).collect()[0][0]
    if failing_rows > 0:
        failed.append(name)
        raise ValueError(f"[DQ FAILED] {name} — {failing_rows:,} failing row(s)")
    passed.append(name)
    print(f"[DQ PASSED] {name}")

In [ ]:
# ── Fact table checks ───────────────────────────────────────────────────────

# No nulls on core fact columns
assert_check(
    "No null keys or values in fact",
    """
    SELECT COUNT(*) FROM gold.fact_monthly_economics
    WHERE month_key    IS NULL
       OR date_key     IS NULL
       OR indicator_key IS NULL
       OR source_key   IS NULL
       OR geography_key IS NULL
       OR value        IS NULL
    """
)

# Grain uniqueness — (month_key, indicator_key) must be unique
assert_check(
    "No duplicate grain keys in fact",
    """
    SELECT COUNT(*) FROM (
        SELECT month_key, indicator_key
        FROM gold.fact_monthly_economics
        GROUP BY month_key, indicator_key
        HAVING COUNT(*) > 1
    )
    """
)

# Referential integrity — date dimension
assert_check(
    "Referential integrity — date_key exists in dim_date",
    """
    SELECT COUNT(*) FROM gold.fact_monthly_economics f
    LEFT JOIN gold.dim_date d ON f.date_key = d.date_key
    WHERE d.date_key IS NULL
    """
)

# Referential integrity — indicator dimension
assert_check(
    "Referential integrity — indicator_key exists in dim_indicator",
    """
    SELECT COUNT(*) FROM gold.fact_monthly_economics f
    LEFT JOIN gold.dim_indicator i ON f.indicator_key = i.indicator_key
    WHERE i.indicator_key IS NULL
    """
)

# Referential integrity — source dimension
assert_check(
    "Referential integrity — source_key exists in dim_source",
    """
    SELECT COUNT(*) FROM gold.fact_monthly_economics f
    LEFT JOIN gold.dim_source s ON f.source_key = s.source_key
    WHERE s.source_key IS NULL
    """
)

# Referential integrity — geography dimension
assert_check(
    "Referential integrity — geography_key exists in dim_geography",
    """
    SELECT COUNT(*) FROM gold.fact_monthly_economics f
    LEFT JOIN gold.dim_geography g ON f.geography_key = g.geography_key
    WHERE g.geography_key IS NULL
    """
)

In [ ]:
# ── Value range checks ──────────────────────────────────────────────────────

# Policy rate (indicator_key=5): Iceland Central Bank rate has never been negative
# or exceeded 30% in the modern era
assert_check(
    "Policy rate within expected range (0–30%)",
    """
    SELECT COUNT(*) FROM gold.fact_monthly_economics
    WHERE indicator_key = 5
      AND (value < 0 OR value > 30)
    """
)

# CPI YoY inflation (indicator_key=6): bounds account for extreme scenarios
assert_check(
    "CPI inflation within expected range (-10% to 100%)",
    """
    SELECT COUNT(*) FROM gold.fact_monthly_economics
    WHERE indicator_key = 6
      AND (value < -10 OR value > 100)
    """
)

# ISK/USD (indicator_key=1) and ISK/EUR (indicator_key=2): exchange rates must be positive
assert_check(
    "ISK exchange rates are positive",
    """
    SELECT COUNT(*) FROM gold.fact_monthly_economics
    WHERE indicator_key IN (1, 2)
      AND value <= 0
    """
)

# OMX index (indicator_key=4): must be positive
assert_check(
    "OMX index is positive",
    """
    SELECT COUNT(*) FROM gold.fact_monthly_economics
    WHERE indicator_key = 4
      AND value <= 0
    """
)

In [ ]:
# ── Completeness checks ─────────────────────────────────────────────────────

# All 7 indicators must be present in the most recent month
assert_check(
    "All 7 indicators present in the latest month",
    """
    SELECT CASE WHEN COUNT(DISTINCT indicator_key) < 7 THEN 1 ELSE 0 END
    FROM gold.fact_monthly_economics
    WHERE month_key = (SELECT MAX(month_key) FROM gold.fact_monthly_economics)
    """
)

# dim_date must cover the full date range present in the fact
assert_check(
    "dim_date covers full fact date range",
    """
    SELECT COUNT(*) FROM gold.fact_monthly_economics f
    WHERE NOT EXISTS (
        SELECT 1 FROM gold.dim_date d
        WHERE d.date_key = f.date_key
    )
    """
)

# Minimum row count — 7 indicators x months since Jan 2022
assert_check(
    "Fact table meets minimum row count",
    """
    SELECT CASE WHEN COUNT(*) < 7 THEN 1 ELSE 0 END
    FROM gold.fact_monthly_economics
    """
)

In [ ]:
# ── Summary ─────────────────────────────────────────────────────────────────
print(f"\nDQ complete — {len(passed)} passed, {len(failed)} failed")
for check in passed:
    print(f"  ✓ {check}")